# T2S Pipeline

Thin orchestrator: every cell just calls into a decoupled module and names its output via `results.py`. No modeling/training logic lives in this notebook — that all lives in the `.py` files, where it's unit-tested (`tests/`).

Expert policy pretraining is **skipped** — assumed already done, checkpoints under `train_res/expertPolicy/`.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import config
import results
config.ensure_dirs()
print('root:', config.ROOT_DIR)

root: d:\!DUKE\2026_Fall\Independent_Research\grl-t2s


## 1. Expert Policy Pretraining

Checkpoints live in `config.EXPERT_POLICY_DIR` = `train_res/expertPolicy/<config.EXPERT_RUN_NAME>/`, alongside `eval_history.json`. Toggle `RUN_EXPERT_TRAINING = False` (default) to just verify an existing run is there; `True` trains into that same folder.

In [2]:
import expert_train, glob

RUN_EXPERT_TRAINING = False   # set True to actually train into config.EXPERT_POLICY_DIR

expert_run_dir = config.EXPERT_POLICY_DIR   # train_res/expertPolicy/<EXPERT_RUN_NAME>/
os.makedirs(expert_run_dir, exist_ok=True)

if RUN_EXPERT_TRAINING:
    results.new_run_dir('expert_policy', config.EXPERT_RUN_NAME, meta={'task_name': config.TASK_NAME})
    model, expert_history = expert_train.train_expert_policy(expert_run_dir)
    print('trained expert, final success rate:', expert_history[-1] if expert_history else None)
else:
    ckpts = glob.glob(os.path.join(expert_run_dir, f'{config.TASK_SLUG}_*.zip'))
    assert ckpts, (f'no expert checkpoints found in {expert_run_dir} for task {config.TASK_NAME} — '
                    'place them there (see chat for the rename command) or set RUN_EXPERT_TRAINING=True')
    print(f'reusing {len(ckpts)} expert checkpoints from {expert_run_dir}')

reusing 8 expert checkpoints from d:\!DUKE\2026_Fall\Independent_Research\grl-t2s\train_res\expertPolicy\peg-insert


## 2. T2S Data Collection

In [3]:
import data_collection
from env_utils import ensure_fixed_task

ensure_fixed_task()

DATA_RUN_NAME = 'v1'
data_run_dir = results.new_run_dir('data_collection', DATA_RUN_NAME,
                                     meta={'notes': 'coupling-based stage detection'})
plan = data_collection.default_collection_plan(expert_policy_dir=expert_run_dir)
summary = data_collection.collect_dataset(data_run_dir, plan=plan)
print(summary)

d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")
d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


{'total_rows': 40000, 'total_episodes': 80, 'obs_dim': 39, 'successful_episodes': 80, 'failed_episodes': 0, 'censor_label': 300.0, 'stage_fractions': {0: 1.0, 1: 0.0, 2: 0.0}, 'sources': ['peg_insert_side_v3_400000.zip_n0.0', 'peg_insert_side_v3_final.zip_n0.0']}


## 3. T2S Model Training (MC / TD(0) / TD(&lambda;))

In [4]:
import t2s_train

T2S_RUN_NAME = 'v1'
t2s_run_dir = results.new_run_dir('t2s_model', T2S_RUN_NAME,
                                    meta={'data_run': DATA_RUN_NAME})
dataset_path = os.path.join(data_run_dir, 'dataset.npz')

models, histories, summary_rows = t2s_train.run_all_combos(
    t2s_run_dir, dataset_path, seeds=(0,))
for row in summary_rows:
    print(row['combo'], 'mean val MSE:', round(row['mean_val_mse'], 1))

mc_succ mean val MSE: 3.5
mc_all mean val MSE: 3.5
td0_succ mean val MSE: 26.1
td0_all mean val MSE: 26.1
tdlambda_succ mean val MSE: 5.4
tdlambda_all mean val MSE: 5.4


## 4. T2S Model Analysis (held-out policies — never in training data)

In [9]:
import json
import t2s_predict, t2s_eval
from stable_baselines3 import SAC

EVAL_RUN_NAME = 'v1'
eval_run_dir = results.new_run_dir('t2s_eval', EVAL_RUN_NAME, meta={'t2s_run': T2S_RUN_NAME})

# TODO: point these at checkpoints that were genuinely never used in data_collection.default_collection_plan()
eval_success_pol = SAC.load('./train_res/expertPolicy/peg-insert/peg_insert_side_v3_700000')
eval_failure_pol = SAC.load('./train_res/expertPolicy/peg-insert/peg_insert_side_v3_100000')

reports = {}
for combo in summary_rows:
    method, condition = combo['combo'].rsplit('_', 1)
    predict_fn = t2s_predict.load_t2s_predictor(t2s_run_dir, method, condition, seed=combo['best_seed'])
    reports[combo['combo']] = t2s_eval.run_full_evaluation(predict_fn, eval_success_pol, eval_failure_pol)

with open(os.path.join(eval_run_dir, 'eval_report.json'), 'w') as f:
    json.dump(reports, f, indent=2)
for combo, r in reports.items():
    print(combo, r.get('success_scenario_summary'))

d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")
d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


mc_succ {'mae': 5.355848891742511, 'rmse': 9.743664252943672, 'pearson_r': 0.9727737966397796, 'spearman_rho': 0.9767152796852736, 'max_wrong_direction': 5.869140625}
mc_all {'mae': 5.355848891742511, 'rmse': 9.743664252943672, 'pearson_r': 0.9727737966397796, 'spearman_rho': 0.9767152796852736, 'max_wrong_direction': 5.869140625}
td0_succ {'mae': 4.721408881963222, 'rmse': 7.3374767779729355, 'pearson_r': 0.986725091856796, 'spearman_rho': 0.9759346319026677, 'max_wrong_direction': 1.2242431640625}
td0_all {'mae': 4.721408881963222, 'rmse': 7.3374767779729355, 'pearson_r': 0.986725091856796, 'spearman_rho': 0.9759346319026677, 'max_wrong_direction': 1.2242431640625}
tdlambda_succ {'mae': 5.3005599506813965, 'rmse': 9.600793030389, 'pearson_r': 0.9679799163041336, 'spearman_rho': 0.9824296214539492, 'max_wrong_direction': 3.9143524169921875}
tdlambda_all {'mae': 5.3005599506813965, 'rmse': 9.600793030389, 'pearson_r': 0.9679799163041336, 'spearman_rho': 0.9824296214539492, 'max_wrong_d

## 5. Downstream RL Training (SAC against the frozen T2S reward)

In [10]:
import policy_train

BEST_COMBO = min(summary_rows, key=lambda r: r['mean_val_mse'])
# the chose the best combo based on the lowest mean validation MSE
print("Best combo:", BEST_COMBO['combo'], "with mean val MSE:", BEST_COMBO['mean_val_mse'])



Best combo: mc_succ with mean val MSE: 3.5142643451690674


In [ ]:
method, condition = BEST_COMBO['combo'].rsplit('_', 1)
predict_fn = t2s_predict.load_t2s_predictor(t2s_run_dir, method, condition, seed=BEST_COMBO['best_seed'])

POLICY_RUN_NAME = f"t2s_{BEST_COMBO['combo']}_absolute_v1"
policy_run_dir = results.new_run_dir('policy', POLICY_RUN_NAME,
                                       meta={'t2s_run': T2S_RUN_NAME, 'combo': BEST_COMBO['combo']})

# train a policy using the best T2S model, Reward Mode: absolute / relative
model, history = policy_train.train_policy(policy_run_dir, predict_fn, reward_mode='absolute',
                                             total_timesteps=1_000_000)
print('final success rate:', history[-1] if history else 'no evals recorded')